# Cibuco_Boriken — BirdCLEF+ 2026 Inference v6 (B2 + Perch + B0)
**Team:** Cibuco_Boriken | **3-Model Ensemble** | **T=0.3** | **234 species**

- **EfficientNet-B2** — CNN on mel spectrograms (largest capacity, best solo)
- **Google Perch Head** — MLP on Perch bird embeddings (maximally diverse features)
- **EfficientNet-B0** — CNN on mel spectrograms (lighter, may catch different patterns)

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q librosa audioread tensorflow tensorflow-hub
print('Dependencies installed ✅')

In [ ]:
# ── Cell 2: Clone repo + add to path ──
!git clone https://github.com/drosadocastro-bit/cibuco-boriken
import sys
sys.path.insert(0, '/kaggle/working/cibuco-boriken')
print('Repo cloned ✅')

In [ ]:
# ── Cell 3: Imports + config ──
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as nnF
import librosa
from pathlib import Path
from tqdm import tqdm

# Paths
DATA_DIR = Path('/kaggle/input/competitions/birdclef-2026')

# ⬇️ UPDATE THESE to your actual Kaggle dataset paths
B2_MODEL_PATH     = Path('/kaggle/input/datasets/drakus74/efficientnet-b2-smartcrop/efficientnet_b2_SMARTCROP_BCE_20260315.pt')
B0_MODEL_PATH     = Path('/kaggle/input/datasets/drakus74/birdclef-julia-model/birdclef_model.pt')  # ⬅️ UPDATE THIS
PERCH_SAVED_MODEL = Path('/kaggle/input/datasets/drakus74/perch-saved-model')  # TF SavedModel dir
PERCH_HEAD_PATH   = Path('/kaggle/input/datasets/drakus74/perch-saved-model/perch_head.pt')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

# Config
SAMPLE_RATE = 32000
WINDOW_SEC  = 5.0
N_MELS      = 128
N_FFT       = 2048
HOP_LENGTH  = 512
FMIN        = 50
FMAX        = 14000
TEMPERATURE = 0.3
BATCH_SIZE  = 32
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

ZERO_SHOT_PRIOR = 1.0 / 234

# Ensemble weights — B2 dominates, Perch adds diversity, B0 adds coverage
W_B2    = 0.55
W_PERCH = 0.30
W_B0    = 0.15

print(f'Device:      {DEVICE}')
print(f'Ensemble:    B2={W_B2}, Perch={W_PERCH}, B0={W_B0}')
print(f'Sum weights: {W_B2 + W_PERCH + W_B0}')
print(f'Temperature: {TEMPERATURE}')
print('Config ready ✅')

In [ ]:
# ── Cell 4: Load species lists ──
taxonomy_df    = pd.read_csv(DATA_DIR / 'taxonomy.csv')
SPECIES        = taxonomy_df['primary_label'].tolist()
NUM_SPECIES    = len(SPECIES)

train_df          = pd.read_csv(DATA_DIR / 'train.csv')
TRAIN_SPECIES     = sorted(train_df['primary_label'].unique().tolist())
NUM_TRAIN_SPECIES = len(TRAIN_SPECIES)

ZERO_SHOT = set(SPECIES) - set(TRAIN_SPECIES)

print(f'Taxonomy: {NUM_SPECIES} | Train: {NUM_TRAIN_SPECIES} | Zero-shot: {len(ZERO_SHOT)}')

In [ ]:
# ── Cell 5: Load all three models ──
from torchvision.models import efficientnet_b2, efficientnet_b0

def load_efficientnet(builder_fn, model_path, num_classes, device):
    """Load an EfficientNet model with _orig_mod prefix handling."""
    model = builder_fn(weights=None)
    in_f = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_f, num_classes),
    )
    state_dict = torch.load(model_path, map_location=device, weights_only=True)
    cleaned = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(cleaned)
    model = model.to(device)
    model.eval()
    return model, in_f

# --- EfficientNet-B2 (1408 features) ---
model_b2, b2_in = load_efficientnet(efficientnet_b2, B2_MODEL_PATH, NUM_TRAIN_SPECIES, DEVICE)
print(f'EfficientNet-B2 loaded ✅  ({b2_in} → {NUM_TRAIN_SPECIES})')

# --- EfficientNet-B0 (1280 features) ---
model_b0, b0_in = load_efficientnet(efficientnet_b0, B0_MODEL_PATH, NUM_TRAIN_SPECIES, DEVICE)
print(f'EfficientNet-B0 loaded ✅  ({b0_in} → {NUM_TRAIN_SPECIES})')

print('CNN models loaded ✅')

In [ ]:
# ── Cell 6: Load Perch (TF) + Perch Head (PyTorch) ──
import tensorflow as tf

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

perch_tf = tf.saved_model.load(str(PERCH_SAVED_MODEL))
print(f'Perch TF model loaded ✅')

class PerchHead(nn.Module):
    def __init__(self, embedding_dim=1280, num_species=206):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_species),
        )
    def forward(self, x):
        return self.classifier(x)

perch_ckpt = torch.load(PERCH_HEAD_PATH, map_location=DEVICE, weights_only=True)
perch_species = perch_ckpt['species']
perch_head = PerchHead(
    embedding_dim=perch_ckpt['embedding_dim'],
    num_species=perch_ckpt['num_species'],
)
perch_head.load_state_dict(perch_ckpt['state_dict'])
perch_head = perch_head.to(DEVICE)
perch_head.eval()

perch_species_idx = {sp: i for i, sp in enumerate(perch_species)}
print(f'Perch Head loaded ✅ ({perch_ckpt["num_species"]} species)')

In [ ]:
# ── Cell 7: Audio + mel + Perch utilities ──
def audio_to_mel(audio, sr=SAMPLE_RATE, n_mels=N_MELS):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=FMIN, fmax=FMAX
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_min, mel_max = mel_db.min(), mel_db.max()
    if mel_max - mel_min > 1e-8:
        mel_db = (mel_db - mel_min) / (mel_max - mel_min)
    else:
        mel_db = np.zeros_like(mel_db)
    return mel_db.astype(np.float32)

def mel_to_tensor(mel):
    t = torch.from_numpy(mel).unsqueeze(0)
    t = t.repeat(3, 1, 1)
    t = nnF.interpolate(
        t.unsqueeze(0), size=(224, 224),
        mode='bilinear', align_corners=False,
    ).squeeze(0)
    return t

def get_perch_embedding(audio_chunk):
    """Extract Perch embedding from a 5s audio chunk."""
    window_samples = int(SAMPLE_RATE * WINDOW_SEC)
    if len(audio_chunk) < window_samples:
        audio_chunk = np.pad(audio_chunk, (0, window_samples - len(audio_chunk)))
    elif len(audio_chunk) > window_samples:
        audio_chunk = audio_chunk[:window_samples]
    waveform = tf.constant(audio_chunk[np.newaxis, :], dtype=tf.float32)
    if hasattr(perch_tf, 'infer_tf'):
        output = perch_tf.infer_tf(waveform)
    elif hasattr(perch_tf, 'front_end'):
        output = perch_tf.front_end(waveform)
    else:
        output = perch_tf(waveform)
    if isinstance(output, dict):
        return output['embedding'].numpy().squeeze()
    elif isinstance(output, (tuple, list)):
        arrays = [o.numpy().squeeze() for o in output]
        return max(arrays, key=lambda a: a.size)
    else:
        return output.numpy().squeeze()

print('Audio + Perch utilities ready ✅')

In [ ]:
# ── Cell 8: Parse sample submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'Sample submission: {len(sample_sub)} rows, {len(sample_sub.columns)} cols')

def parse_row_id(row_id):
    parts = row_id.rsplit('_', 1)
    return parts[0], int(parts[1])

from collections import defaultdict
file_windows = defaultdict(list)
for row_id in sample_sub['row_id']:
    stem, end_time = parse_row_id(row_id)
    file_windows[stem].append((row_id, end_time))

print(f'Soundscape files: {len(file_windows)}')
print('Row IDs parsed ✅')

In [ ]:
# ── Cell 9: Run B2 + Perch + B0 ensemble inference ──
soundscape_dir = DATA_DIR / 'test_soundscapes'
train_species_idx = {sp: i for i, sp in enumerate(TRAIN_SPECIES)}

all_rows = {}

for file_stem in tqdm(sorted(file_windows.keys()), desc='B2+Perch+B0 Ensemble'):
    audio_path = None
    for ext in ['.ogg', '.wav', '.flac']:
        candidate = soundscape_dir / f'{file_stem}{ext}'
        if candidate.exists():
            audio_path = candidate
            break

    if audio_path is None:
        for row_id, end_time in file_windows[file_stem]:
            row = {'row_id': row_id}
            for sp in SPECIES:
                row[sp] = ZERO_SHOT_PRIOR
            all_rows[row_id] = row
        continue

    try:
        audio, _ = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f'Error loading {audio_path}: {e}')
        for row_id, end_time in file_windows[file_stem]:
            row = {'row_id': row_id}
            for sp in SPECIES:
                row[sp] = ZERO_SHOT_PRIOR
            all_rows[row_id] = row
        continue

    window_samples = int(WINDOW_SEC * SAMPLE_RATE)
    windows_info = file_windows[file_stem]
    mel_tensors = []
    perch_embs = []
    row_ids_order = []

    for row_id, end_time in windows_info:
        start_sample = int((end_time - WINDOW_SEC) * SAMPLE_RATE)
        end_sample   = int(end_time * SAMPLE_RATE)
        start_sample = max(0, start_sample)
        chunk = audio[start_sample:end_sample]
        if len(chunk) < window_samples:
            chunk = np.pad(chunk, (0, window_samples - len(chunk)))

        mel = audio_to_mel(chunk)
        mel_tensors.append(mel_to_tensor(mel))

        emb = get_perch_embedding(chunk)
        perch_embs.append(emb)

        row_ids_order.append(row_id)

    # Stack batches
    mel_batch_all = torch.stack(mel_tensors).to(DEVICE)
    perch_batch_all = torch.from_numpy(np.stack(perch_embs)).float().to(DEVICE)

    all_probs_b2 = []
    all_probs_b0 = []
    all_probs_perch = []

    for i in range(0, len(mel_batch_all), BATCH_SIZE):
        mel_b = mel_batch_all[i:i+BATCH_SIZE]
        perch_b = perch_batch_all[i:i+BATCH_SIZE]

        with torch.no_grad():
            # B2
            logits_b2 = model_b2(mel_b)
            probs_b2 = torch.sigmoid(logits_b2 / TEMPERATURE).cpu().numpy()

            # B0 — same mel input, different model
            logits_b0 = model_b0(mel_b)
            probs_b0 = torch.sigmoid(logits_b0 / TEMPERATURE).cpu().numpy()

            # Perch
            logits_perch = perch_head(perch_b)
            probs_perch = torch.sigmoid(logits_perch / TEMPERATURE).cpu().numpy()

        all_probs_b2.append(probs_b2)
        all_probs_b0.append(probs_b0)
        all_probs_perch.append(probs_perch)

    probs_b2 = np.vstack(all_probs_b2)
    probs_b0 = np.vstack(all_probs_b0)
    probs_perch = np.vstack(all_probs_perch)

    for idx, row_id in enumerate(row_ids_order):
        row = {'row_id': row_id}
        for sp in SPECIES:
            p_b2    = float(probs_b2[idx, train_species_idx[sp]])    if sp in train_species_idx else ZERO_SHOT_PRIOR
            p_b0    = float(probs_b0[idx, train_species_idx[sp]])    if sp in train_species_idx else ZERO_SHOT_PRIOR
            p_perch = float(probs_perch[idx, perch_species_idx[sp]]) if sp in perch_species_idx else ZERO_SHOT_PRIOR
            row[sp] = W_B2 * p_b2 + W_PERCH * p_perch + W_B0 * p_b0
        all_rows[row_id] = row

print(f'Total rows: {len(all_rows)} (expected: {len(sample_sub)}) ✅')

In [ ]:
# ── Cell 10: Generate submission.csv ──
rows_ordered = [all_rows[rid] for rid in sample_sub['row_id']]
submission_df = pd.DataFrame(rows_ordered)
submission_df = submission_df[sample_sub.columns]
submission_df = submission_df.fillna(ZERO_SHOT_PRIOR)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f'Submission saved ✅')
print(f'Shape: {submission_df.shape} (expected: {sample_sub.shape})')
print(f'Min prob: {submission_df.iloc[:, 1:].min().min():.6f}')
print(f'Max prob: {submission_df.iloc[:, 1:].max().max():.6f}')
print(submission_df.head(3))

In [ ]:
# ── Cell 11: Validate submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
our_sub    = pd.read_csv(OUTPUT_PATH)

print('=== Submission Validation ===')
print(f'Expected rows:  {len(sample_sub)}')
print(f'Our rows:       {len(our_sub)}')
print(f'Expected cols:  {len(sample_sub.columns)}')
print(f'Our cols:       {len(our_sub.columns)}')
print(f'Columns match:  {set(sample_sub.columns) == set(our_sub.columns)}')
print(f'Row IDs match:  {set(sample_sub["row_id"]) == set(our_sub["row_id"])}')
print(f'No NaN:         {our_sub.isna().sum().sum() == 0}')
print(f'No zeros:       {(our_sub.iloc[:, 1:] == 0).sum().sum() == 0}')
print()
if set(sample_sub.columns) == set(our_sub.columns):
    print('✅ B2+PERCH+B0 ENSEMBLE VALID — ready to submit!')
else:
    missing = set(sample_sub.columns) - set(our_sub.columns)
    extra   = set(our_sub.columns) - set(sample_sub.columns)
    if missing: print(f'⚠️  Missing columns: {missing}')
    if extra:   print(f'⚠️  Extra columns: {extra}')